# 24 · GPU memory hierarchy and picking the cheapest GPU
Level L4 · Part 3 · core · ~20 min · builds on 06, 19

## Goal
By the end of this lesson you can compare L4, A100, H100 and H200 by memory, memory bandwidth and FP8 support, find the smallest group of each card that holds a 70B model's weights, and pick the cheapest group once you enter a price.

## Why it matters
- **Without it:** you choose a GPU by its name, or by the lowest price per card. The 70B model then fails to load on one card, or it loads and decodes slowly, and the bill covers the wrong number of GPUs.
- **With it:** three numbers from the spec sheet (memory, bandwidth, FP8 support) and the model size give you the group size and a speed floor for each card. The group's price per hour then decides.
- **When you don't need it:** if a hosted API serves the model, the card is not your choice. You need this before you reserve GPUs, or to check whether a latency target is physically possible.

## The idea
Two GPU numbers matter most for LLM serving. **Memory** decides whether the weights fit and how much room is left for the KV cache. **Memory bandwidth** decides how fast the GPU can read those weights, and decode reads all of them for every token (lesson 06). **FP8 support** decides whether 1-byte numbers run on the fast FP8 tensor cores. Think of a warehouse with a forklift: floor space is memory, and forklift speed is bandwidth. A huge warehouse with a slow forklift stores a lot but ships each order slowly. The cheapest GPU is not the cheapest card. It is the lowest price for the smallest group that fits, with room left for the cache.

## Picture
```mermaid
flowchart TB
  S["On-chip SRAM: small and fastest, too small for weights"] --> H["GPU memory (HBM or GDDR): weights and KV cache live here"]
  H -->|"bandwidth: 0.3 to 4.8 TB/s, read every decode step"| C["compute units: FLOP/s"]
  H -->|"PCIe link: slower, used for offload (lesson 13)"| R["CPU RAM and storage"]
```

## Explained
**The memory hierarchy.** A GPU has a small, very fast on-chip memory, a large GPU memory (HBM on the data-centre cards, GDDR on the L4) and a slower link to CPU RAM. The weights of a 70B model are far too big for the on-chip memory, so each decode step streams them from GPU memory.

**Dense or sparse.** NVIDIA's pages quote sparse FLOP/s, which is twice the dense number. This course uses dense numbers, the ones a dense model gets.

**The four cards.** The table is built from NVIDIA's product pages (opened 2026-10-21, RESEARCH.md). The ridge point is dense FLOP/s divided by bandwidth (lesson 06). The last column is the Google Cloud machine family.

In [1]:
GPUS = {                          # NVIDIA product pages, opened 2026-10-21; dense FP16 = sparse / 2
    "L4":        {"mem_gb": 24,  "bw_gbs": 300,  "tflops": 121, "fp8": True,  "gcp": "G2 (g2-standard)"},
    "A100 80GB": {"mem_gb": 80,  "bw_gbs": 2039, "tflops": 312, "fp8": False, "gcp": "A2 Ultra"},
    "H100 80GB": {"mem_gb": 80,  "bw_gbs": 3350, "tflops": 989, "fp8": True,  "gcp": "A3 High"},
    "H200":      {"mem_gb": 141, "bw_gbs": 4800, "tflops": 989, "fp8": True,  "gcp": "A3 Ultra"},
}

print(f"{'GPU':10} {'memory':>7} {'GB/s':>6} {'TFLOP/s':>8} {'FP8':>4} {'ridge':>6}  GCP family")
for name, g in GPUS.items():
    ridge = g["tflops"] * 1e12 / (g["bw_gbs"] * 1e9)
    print(f"{name:10} {g['mem_gb']:>4} GB {g['bw_gbs']:>6,} {g['tflops']:>8} "
          f"{'yes' if g['fp8'] else 'no':>4} {ridge:>6.0f}  {g['gcp']}")

GPU         memory   GB/s  TFLOP/s  FP8  ridge  GCP family
L4           24 GB    300      121  yes    403  G2 (g2-standard)
A100 80GB    80 GB  2,039      312   no    153  A2 Ultra
H100 80GB    80 GB  3,350      989  yes    295  A3 High
H200        141 GB  4,800      989  yes    206  A3 Ultra


**What you should see:** the L4 has the least memory and bandwidth and the highest ridge point (about 403 FLOP per byte). The H200 has the most memory and the highest bandwidth. The H100 and H200 share the same dense FLOP/s. The A100 has no FP8 tensor cores.

**The speed floor.** At batch 1, one decode step cannot take less time than the weight bytes divided by bandwidth. The next cell uses an 8B model in FP16 (16 GB of weights, lesson 06).

In [2]:
WEIGHTS_8B_FP16 = 16e9            # Llama 3.1 8B in FP16 (lesson 06)
for name, g in GPUS.items():
    floor_ms = WEIGHTS_8B_FP16 / (g["bw_gbs"] * 1e9) * 1000
    print(f"{name:10} decode floor {floor_ms:5.1f} ms per token at batch 1")

L4         decode floor  53.3 ms per token at batch 1
A100 80GB  decode floor   7.8 ms per token at batch 1
H100 80GB  decode floor   4.8 ms per token at batch 1
H200       decode floor   3.3 ms per token at batch 1


**What you should see:** 53.3 ms on the L4, 7.8 ms on the A100, 4.8 ms on the H100 and 3.3 ms on the H200. The H200 is faster than the H100 because of its bandwidth, not its FLOP/s. Batching shares one weight read across many requests (lesson 12), so these numbers are per-stream floors, not capacity.

**FP8 support.** The L4, H100 and H200 have FP8 tensor cores. The A100 does not. On the A100 you can still store FP8 weights through weight-only kernels, but the maths does not run in FP8 on tensor cores, so do not promise a speedup from FP8 there. Lesson 20 measures it.

## Worked example: the smallest group that fits a 70B model
Vendor memory is not all free. vLLM's default `gpu_memory_utilization` of 0.92 keeps 8% of each card for other uses, so one card gives the engine `memory × 0.92`. The 70B model has 70.55 billion parameters: 131.4 GiB in FP16 and 65.7 GiB in FP8. The next cell finds, for each card and format, the smallest group of 1, 2, 4 or 8 GPUs (the degrees that divide 64 query heads and 8 KV heads) whose weights leave room for the cache.

In [3]:
GIB = 2**30
UTIL = 0.92                       # vLLM default gpu_memory_utilization
PARAMS_70B = 70_553_706_496       # Llama 3.1 70B parameters (lesson 19)
KV_PER_TOKEN = 327_680            # 320 KiB per token, FP16 cache (lesson 03)
FORMATS = {"FP16": 2, "FP8": 1}   # bytes per weight

def smallest_group(name, bytes_per_param):
    """Smallest tensor-parallel group whose per-GPU weights leave room for the cache, and that room."""
    usable = GPUS[name]["mem_gb"] * 1e9 * UTIL
    for tp in (1, 2, 4, 8):
        room = usable - PARAMS_70B * bytes_per_param / tp
        if room > 0:
            return tp, room
    return None, None

for name in GPUS:
    for fmt, b in FORMATS.items():
        tp, room = smallest_group(name, b)
        if tp is None:
            print(f"{name:10} {fmt}: no group up to 8 fits")
        else:
            tokens = room * tp / KV_PER_TOKEN
            print(f"{name:10} {fmt}: TP={tp}, room {room / GIB:5.2f} GiB per GPU, "
                  f"{tokens:,.0f} KV tokens for the group")

L4         FP16: TP=8, room  4.14 GiB per GPU, 108,437 KV tokens for the group
L4         FP8: TP=4, room  4.14 GiB per GPU, 54,218 KV tokens for the group
A100 80GB  FP16: TP=2, room  2.84 GiB per GPU, 18,593 KV tokens for the group
A100 80GB  FP8: TP=1, room  2.84 GiB per GPU, 9,297 KV tokens for the group
H100 80GB  FP16: TP=2, room  2.84 GiB per GPU, 18,593 KV tokens for the group
H100 80GB  FP8: TP=1, room  2.84 GiB per GPU, 9,297 KV tokens for the group
H200       FP16: TP=2, room 55.10 GiB per GPU, 361,122 KV tokens for the group
H200       FP8: TP=1, room 55.10 GiB per GPU, 180,561 KV tokens for the group


**What you should see:**
- L4: FP16 needs TP=8 and FP8 needs TP=4, each with 4.14 GiB of room per GPU.
- A100 80GB and H100 80GB: FP16 needs TP=2, with only 2.84 GiB of room per GPU. FP8 fits on one card, but leaves 2.84 GiB, about 9,300 tokens of cache (roughly five Brightside requests of 1,800 tokens).
- H200: FP16 needs TP=2 and FP8 fits on one card, with 55.10 GiB of room. That is about 180,000 cache tokens on one GPU.

"Room" is counted before activations and the CUDA workspace, so it is an upper bound. Next, the room at every group size for one H100 in FP16.

In [4]:
usable_h100 = GPUS["H100 80GB"]["mem_gb"] * 1e9 * UTIL
for tp in (2, 4, 8):
    room = usable_h100 - PARAMS_70B * 2 / tp
    print(f"H100 FP16, TP={tp}: room {room / GIB:5.2f} GiB per GPU")

H100 FP16, TP=2: room  2.84 GiB per GPU
H100 FP16, TP=4: room 35.69 GiB per GPU
H100 FP16, TP=8: room 52.12 GiB per GPU


**What you should see:** TP=2 leaves 2.84 GiB, TP=4 leaves 35.69 GiB and TP=8 leaves 52.12 GiB. The smallest group that fits is not always the one you should run. A group of two H100s holds the model but has little cache room.

### Pick the cheapest group
A group's cost per hour is the price of one GPU times the group size. Enter the on-demand price per GPU-hour from your own billing catalog or price list. The course does not supply prices: the third-party numbers we found disagree by about 2x, so none are used here. Leave a price as `None` until you have checked it.

In [5]:
# Enter your prices per GPU-hour in USD. None means "not entered yet".
PRICE_PER_GPU_HOUR = {"L4": None, "A100 80GB": None, "H100 80GB": None, "H200": None}

def cheapest_group(bytes_per_param):
    """Cheapest cost per hour for one copy of the model, over the cards you have priced."""
    options = []
    for name in GPUS:
        tp, _ = smallest_group(name, bytes_per_param)
        if tp is None or PRICE_PER_GPU_HOUR[name] is None:
            continue
        options.append((PRICE_PER_GPU_HOUR[name] * tp, name, tp))
    return min(options) if options else None

for fmt, b in FORMATS.items():
    best = cheapest_group(b)
    if best is None:
        print(f"{fmt}: enter at least one price above, then rerun this cell")
    else:
        cost, name, tp = best
        print(f"{fmt}: {name}, group of {tp}, {cost:.2f} USD per hour for one copy")

FP16: enter at least one price above, then rerun this cell
FP8: enter at least one price above, then rerun this cell


**What you should see:** until you enter prices, both lines ask for one. That is on purpose: a made-up price would give a made-up answer. Compare all four cards before you decide, because the function only looks at the cards you priced. One copy is one replica. More traffic needs more replicas (lesson 26), and cost per million tokens comes in lesson 28.

## At work
- **Databricks** defines model bandwidth utilisation (MBU) as achieved bandwidth divided by peak. Its example reads 14 GB of FP16 weights in 14 ms per token, 1 TB/s, on a 2 TB/s peak: 50%. Measure decode bandwidth. → lesson 07.
- **FP8 on A100:** the A100 has no FP8 tensor cores, so an FP8 plan that assumes a speedup there will miss its target. → lesson 20.
- **Cloudflare (May 2026)** says Kimi K2.5, about 560 GB of weights, needs 8 H100 GPUs. You buy the group that holds one copy. → lesson 28.

Sources: Databricks, "LLM Inference Performance Engineering: Best Practices": https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices · Cloudflare, "High-performance LLMs": https://blog.cloudflare.com/high-performance-llms/ · NVIDIA product pages, opened 2026-10-21 (RESEARCH.md).

## Common misconceptions
1. **"The biggest memory wins."** More memory only helps when the weights or the cache need it. An 8B model in FP16 fits on one L4 with room to spare (lesson 19).
2. **"TFLOP/s sets how fast the answer streams."** For decode, bandwidth sets the floor. The H100 and H200 have the same dense FLOP/s, but the H200's bandwidth makes its floor about 1.4 times lower.
3. **"FP8 speeds up every card."** Only the cards with FP8 tensor cores (L4, H100, H200) run FP8 maths faster.
4. **"If the weights fit, the card can serve."** The room after the weights sets how many requests you can hold (lesson 05). Nine thousand cache tokens is only about five Brightside requests.

## Check yourself
1. Your team has one A100 80GB and wants the FP8 70B model on one card. Does it fit, and how much cache room is left per GPU before activations?
2. An 8B model in FP16 decodes at batch 1. Which card has the lower floor, the H100 or the H200, and by about how much?
3. A colleague says "FP8 gives us 2x on our A100s." What do you check first?
4. Why does the L4 need TP=8 for FP16 while the H100 needs only TP=2?
5. Your team picks the H200 because it has the most memory. Which two numbers do you check before you agree?

1. Yes. The FP8 weights are 65.7 GiB and 68.55 GiB is usable, so 2.84 GiB is left for the cache, about 9,300 tokens before activations.
2. The H200. Its floor is 3.3 ms per token against 4.8 ms, about 1.4 times faster, because its bandwidth is 4.8 TB/s against 3.35 TB/s.
3. Whether the card has FP8 tensor cores. The A100 does not, so measure the speedup on your card (lesson 20) before you promise it.
4. The L4 has 20.56 GiB usable, far below 131.4 GiB of FP16 weights, so it needs eight shards. The H100 has 68.55 GiB, so two shards of 65.7 GiB fit.
5. The group price for the format you serve (price per GPU times group size) and the decode floor from bandwidth. Extra memory matters only if the fit or cache room needs it.

## Recap
- Memory decides whether the weights fit and how much cache room is left. Bandwidth sets the decode floor per token.
- At 0.92 utilisation: one L4 gives 20.56 GiB, one 80 GB card 68.55 GiB, one H200 120.8 GiB. The 70B FP16 weights need 131.4 GiB.
- The cheapest option is the lowest price for the smallest group that fits, with room left. Enter your own prices.
- FP8 tensor cores: L4, H100 and H200, not A100.

**Interview:** "Which GPU would you pick for a 70B model in FP16?" Strong answer: the weights are 141 GB, so no single L4, A100 or H100 holds them with cache room. Two H100s hold them barely; four leave real room. Then compare the group's price per hour with the decode floor and the latency target.